# Day 6 — LangChain Fundamentals
---
> **Phase:** 1 — LLM Foundations  
> **Status:** ✅ Complete

## 🧠 Theory — Read This Before Touching Code

### 1. Why Does LangChain Exist?

Before LangChain, every developer writing LLM-powered applications had to solve the same problems from scratch:

- How do I format a prompt with variables?
- How do I parse the LLM's response into something usable?
- How do I chain multiple LLM calls together?
- How do I manage conversation memory?
- How do I switch between OpenAI, Groq, Anthropic without rewriting everything?

Every team was writing the same glue code independently. LangChain emerged to standardize all of this.

**The core idea:** LangChain defines a common interface for LLM components — prompts, models, parsers, memory, tools — so they can be composed together like LEGO bricks regardless of which provider you use underneath.

```
Without LangChain:
  You → Groq API → parse response manually → format next prompt manually → repeat

With LangChain:
  You → prompt_template | llm | output_parser → done
```

The pipe `|` operator is called **LCEL** (LangChain Expression Language). We will cover it in detail shortly.

**Important mental model:** LangChain is an abstraction layer. It does NOT add intelligence. It adds structure. Everything LangChain does, you could do manually — you proved this in Days 1–5. LangChain just makes it cleaner, faster, and interoperable.

### 2. The Message System — Mapping to What You Already Know

In Day 2, you built a chatbot using raw role/content dictionaries:

```python
# Day 2 — raw format
{"role": "system",    "content": "You are a helpful assistant."}
{"role": "user",      "content": "What is a token?"}
{"role": "assistant", "content": "A token is..."}
```

LangChain wraps these exact same concepts in Python objects:

```python
# Day 6 — LangChain format
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

SystemMessage(content="You are a helpful assistant.")  # role: system
HumanMessage(content="What is a token?")               # role: user
AIMessage(content="A token is...")                     # role: assistant
```

**Why objects instead of dicts?**
- Objects can carry metadata (token counts, model name, timestamps)
- Objects have type safety — you cannot accidentally pass the wrong message type
- LangChain components know how to serialize these to any provider's format automatically

When you use `ChatGroq` (LangChain's Groq wrapper), it internally converts these message objects back to the `{"role": ..., "content": ...}` format that Groq expects. You never see that conversion — it just happens.

This is the abstraction benefit: write once in LangChain's message format, run on any provider.

### 3. ChatGroq — The Standardized LLM Wrapper

In Day 1, you used the raw Groq SDK:

```python
# Day 1 — raw Groq SDK
from groq import Groq
client = Groq(api_key=...)
response = client.chat.completions.create(model=..., messages=..., temperature=...)
text = response.choices[0].message.content  # manual extraction
```

LangChain's `ChatGroq` wraps this in a standardized interface:

```python
# Day 6 — LangChain ChatGroq
from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.7)
response = llm.invoke([HumanMessage(content="What is a token?")])
# response is an AIMessage object — no manual extraction needed
```

**The key benefit:** If tomorrow you want to switch from Groq to OpenAI, you only change one line:

```python
# Before
from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.1-8b-instant")

# After — literally just change this
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o")

# Everything else downstream stays IDENTICAL
```

In production, this matters enormously. Provider outages happen. Pricing changes. Models get deprecated. LangChain's standardized interface means your application logic is completely decoupled from your provider choice.

### 4. ChatPromptTemplate — Reusable Prompt Blueprints

In Day 3, every time you wanted a different prompt, you manually rewrote the string. That works for scripts. It does not work for applications where the same prompt structure is reused with different inputs.

**The problem with hardcoded prompts:**

```python
# Fragile — hardcoded, cannot reuse
messages = [
    {"role": "system", "content": "You are an expert in Python."},
    {"role": "user",   "content": "Explain decorators to a beginner."}
]
```

What if you want to change the topic? Or the expertise? You rewrite the string every time.

**ChatPromptTemplate solves this:**

```python
from langchain_core.prompts import ChatPromptTemplate

# Define the structure ONCE with {placeholders}
template = ChatPromptTemplate.from_messages([
    ("system", "You are an expert in {subject}."),
    ("human",  "Explain {topic} to a {audience}.")
])

# Fill in values at call time
filled = template.invoke({
    "subject":  "Python",
    "topic":    "decorators",
    "audience": "beginner"
})
```

**What `invoke()` returns:** A `ChatPromptValue` object — a list of formatted message objects ready to be sent to the LLM.

**Why this is powerful in production:**
- Define prompt once → reuse across hundreds of calls with different inputs
- Easy to test — just test the template separately from the LLM call
- Version-controllable — prompt templates become first-class code artifacts
- The `{placeholder}` syntax is validated — missing variables raise clear errors

### 5. LCEL — The Pipe Operator

This is the most important concept in modern LangChain. Everything builds on it.

**LCEL = LangChain Expression Language**

It uses Python's `|` (bitwise OR) operator — but LangChain overrides it to mean **chain these components together**.

```python
chain = prompt_template | llm | output_parser
```

Read this exactly as a data pipeline flowing left to right:
1. Input dict goes into `prompt_template` → produces formatted messages
2. Formatted messages go into `llm` → produces an `AIMessage`
3. `AIMessage` goes into `output_parser` → produces plain text or a dict

**To run the chain:**
```python
result = chain.invoke({"subject": "Python", "topic": "decorators", "audience": "beginner"})
```

**Why LCEL over function calls?**

You could manually write:
```python
# Manual approach — verbose
formatted = prompt_template.invoke(inputs)
ai_message = llm.invoke(formatted)
result = output_parser.invoke(ai_message)
```

LCEL compresses this to:
```python
# LCEL — clean
result = (prompt_template | llm | output_parser).invoke(inputs)
```

But it is not just syntactic sugar. LCEL chains automatically get:
- **Streaming support** — the chain can stream tokens as they are generated
- **Async support** — `chain.ainvoke()` works automatically
- **Batch support** — `chain.batch([input1, input2, ...])` works automatically
- **LangSmith tracing** — every step is automatically logged for observability

You get all of that for free just by using `|` instead of manual calls.

### 6. Output Parsers — Extracting What You Actually Want

When you call `.invoke()` on a `ChatGroq` model, it returns an `AIMessage` object:

```python
AIMessage(
    content="A token is a unit of text...",
    response_metadata={"token_usage": {...}, "model_name": "llama-3.1-8b-instant"},
    id="run-abc123"
)
```

Usually you just want the text string. Output parsers handle that extraction.

#### StrOutputParser

```python
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()
# Takes AIMessage → returns plain string
# AIMessage(content="A token is...") → "A token is..."
```

#### JsonOutputParser

```python
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()
# Takes AIMessage with JSON content → returns Python dict
# AIMessage(content='{"sentiment": "positive"}') → {"sentiment": "positive"}
```

This replaces the manual `json.loads(response.choices[0].message.content)` pattern you used in Day 3.

**Why parsers as separate components?**
Because in LCEL, every component has one job. The LLM generates text. The parser transforms it. Keeping them separate means you can swap parsers without touching the rest of the chain.

### 7. Memory — How LangChain Solves the Stateless Problem

You already understand from Day 2 that LLMs are stateless. The model remembers nothing between calls. Your code has to manage history manually — appending user messages, appending assistant replies, sending the full list every time.

In Day 2 you did this:

```python
conversation_history = [{"role": "system", "content": "..."}]

# Every turn:
conversation_history.append({"role": "user",      "content": user_input})
response = client.chat.completions.create(messages=conversation_history)
conversation_history.append({"role": "assistant", "content": response.choices[0].message.content})
```

LangChain automates exactly this with memory components.

#### InMemoryChatMessageHistory

```python
from langchain_core.chat_history import InMemoryChatMessageHistory

history = InMemoryChatMessageHistory()
history.add_user_message("My name is Sara.")
history.add_ai_message("Hello Sara! How can I help?")
# history.messages → list of HumanMessage and AIMessage objects
```

This is your `conversation_history` list from Day 2 — just a cleaner interface.

#### Session IDs — Why They Matter

Imagine building a customer support chatbot with 1000 simultaneous users. If you use one shared memory object, everyone's conversations get mixed together. Session IDs solve this:

```python
# Each user gets their own isolated memory
store = {}  # dictionary mapping session_id → history object

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# User A
get_session_history("user_a").add_user_message("I am Sara.")

# User B
get_session_history("user_b").add_user_message("I am Ravi.")

# Completely isolated — Sara's history never leaks to Ravi
```

In production systems this `store` would be a database or Redis cache instead of a Python dict. But the pattern is identical.

#### MessagesPlaceholder — Injecting History Into Prompts

For memory to work with LCEL chains, the prompt template needs a slot where history gets injected:

```python
from langchain_core.prompts import MessagesPlaceholder

template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),  # ← history injected here
    ("human", "{input}")
])
```

At runtime the prompt becomes:
```
[SystemMessage: You are a helpful assistant.]
[HumanMessage: My name is Sara.]        ← from history
[AIMessage: Hello Sara!]                ← from history
[HumanMessage: What did I tell you?]    ← current input
```

This is exactly the same structure as your Day 2 manual history list — LangChain just assembles it automatically.

#### RunnableWithMessageHistory — Gluing It All Together

```python
from langchain_core.runnables.history import RunnableWithMessageHistory

chain_with_memory = RunnableWithMessageHistory(
    chain,                              # your LCEL chain
    get_session_history,                # function that returns history for a session
    input_messages_key="input",         # which key in your input dict is the user message
    history_messages_key="history"      # which placeholder in your template to fill
)

# To invoke:
chain_with_memory.invoke(
    {"input": "What did I tell you my name was?"},
    config={"configurable": {"session_id": "user_sara"}}
)
```

**What `RunnableWithMessageHistory` does automatically on every call:**
1. Loads history from `get_session_history(session_id)`
2. Injects it into the `MessagesPlaceholder`
3. Runs the chain
4. Saves the new human + AI messages back to history

Your entire Day 2 chatbot loop — the append/send/append pattern — is now handled in one wrapper.

### 8. The Complete Mental Model — Day 2 vs Day 6

This table shows exactly what LangChain replaces and why it is better:

| Day 2 Raw Approach | Day 6 LangChain Equivalent | What Improved |
|---|---|---|
| `{"role": "user", "content": ...}` | `HumanMessage(content=...)` | Type safety, metadata |
| `Groq(api_key=...)` + manual call | `ChatGroq(model=..., temperature=...)` | Provider-agnostic |
| Manual f-string prompt building | `ChatPromptTemplate` with `{variables}` | Reusable, testable |
| `response.choices[0].message.content` | `StrOutputParser()` | Clean, chainable |
| `json.loads(response...)` | `JsonOutputParser()` | Clean, chainable |
| Manual append/send/append loop | `RunnableWithMessageHistory` | Automated, session-aware |
| One global history list | Session ID + store dict | Multi-user safe |

## 💻 Code

### Install Required Packages

```bash
uv pip install langchain langchain-groq langchain-community langchain-huggingface
```

In [ ]:
# --- Part 1: Basic Chain with LCEL ---
# prompt_template | llm | output_parser

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os

load_dotenv()

# The LLM wrapper — standardized interface over raw Groq SDK
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.7,
    api_key=os.getenv("GROQ_API_KEY")
)

# Reusable prompt template with variables
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert in {subject}. Explain clearly and concisely."),
    ("human",  "Explain {topic} to a {audience}.")
])

# Output parser — extracts plain string from AIMessage
parser = StrOutputParser()

# Build the chain using LCEL pipe operator
chain = prompt | llm | parser

# Run it
result = chain.invoke({
    "subject":  "Python",
    "topic":    "decorators",
    "audience": "complete beginner"
})

print(result)

In [ ]:
# --- Part 2: JSON Output Chain ---
# Same as Day 3 sentiment pipeline — now clean LCEL

from langchain_core.output_parsers import JsonOutputParser

llm_zero_temp = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    api_key=os.getenv("GROQ_API_KEY")
)

sentiment_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a sentiment analysis engine.
Return only valid raw JSON with no markdown, no backticks, no explanation.
Return exactly these four fields:
- sentiment: must be exactly one of: Positive, Negative, Neutral
- reasoning: one sentence explaining the classification
- confidence: float between 0 and 1
- summary: one line summary of the review"""),
    ("human", "{review}")
])

json_parser = JsonOutputParser()

sentiment_chain = sentiment_prompt | llm_zero_temp | json_parser

result = sentiment_chain.invoke({
    "review": "The product arrived late but worked perfectly."
})

print("Sentiment:  ", result["sentiment"])
print("Confidence: ", result["confidence"])
print("Reasoning:  ", result["reasoning"])
print("Summary:    ", result["summary"])

In [ ]:
# --- Part 3: Chatbot with Memory using RunnableWithMessageHistory ---

from langchain_core.prompts import MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# Step 1: Build the base chain (no memory yet)
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),   # history injected here
    ("human", "{input}")
])

base_chain = chat_prompt | llm | StrOutputParser()

# Step 2: Session store — maps session_id to history object
store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# Step 3: Wrap chain with memory
chain_with_memory = RunnableWithMessageHistory(
    base_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

# Step 4: Run the chatbot loop
print("Chatbot ready (LangChain). Type 'quit' to exit.\n")

session_id = "user_sara"

while True:
    user_input = input("You: ")
    
    if user_input.lower() == "quit":
        break
    
    response = chain_with_memory.invoke(
        {"input": user_input},
        config={"configurable": {"session_id": session_id}}
    )
    
    print(f"\nBot: {response}\n")

## 🔬 Key Observations

| What You Tried | What You Observed | The Lesson |
|---|---|---|
| `chain.invoke()` vs raw `client.chat.completions.create()` | Same result, much less code | LCEL reduces boilerplate |
| JsonOutputParser | Returns Python dict directly | No manual `json.loads()` needed |
| Multiple turns in memory chatbot | Bot remembers name across turns | `RunnableWithMessageHistory` handles append/send/append loop |
| Different session IDs | Conversations fully isolated | Session store pattern works |
| Swapping StrOutputParser for JsonOutputParser | Only one line changes | LCEL components are modular |

## ✅ Day 6 Summary

```
✓ LangChain exists to standardize and compose LLM components
✓ ChatGroq — provider-agnostic LLM wrapper over raw Groq SDK
✓ HumanMessage, SystemMessage, AIMessage — typed wrappers over role/content dicts
✓ ChatPromptTemplate — reusable multi-variable prompt blueprints
✓ LCEL pipe operator — chains components: prompt | llm | parser
✓ StrOutputParser — extracts plain string from AIMessage
✓ JsonOutputParser — parses JSON string directly into Python dict
✓ InMemoryChatMessageHistory — LangChain's memory store (same as Day 2 list)
✓ Session IDs — isolate conversations per user in multi-user systems
✓ MessagesPlaceholder — injects history into prompt template
✓ RunnableWithMessageHistory — automates the append/send/append loop
✓ The entire Day 2 chatbot pattern is now 3 objects + 1 wrapper
```

## ⚠️ Common Mistakes to Avoid

1. **Forgetting `MessagesPlaceholder`** — if you add memory wrapper but no placeholder in the template, history has nowhere to be injected and gets silently ignored
2. **Wrong `input_messages_key`** — must match the key name in your `.invoke()` dict exactly
3. **Wrong `history_messages_key`** — must match the `variable_name` in your `MessagesPlaceholder` exactly
4. **Using temperature > 0 for JSON output** — always `temperature=0` for structured output chains, same rule as Day 3
5. **Expecting `JsonOutputParser` to fix bad prompts** — the prompt still needs to instruct the model to return raw JSON with no backticks

## 🔜 Coming in Day 7 — LlamaIndex

LangChain gave you cleaner abstractions for the same things you built in Days 1–3.
LlamaIndex specializes specifically in the data-to-LLM problem — loading documents, chunking them, indexing them, and querying them.

You already built the retrieval engine from scratch in Days 4–5.
In Day 7 you will see how LlamaIndex wraps exactly that same pipeline with a cleaner API — and understand what it is doing under the hood because you have done it manually.

**Pre-Day 7 Question:**
In Day 5 your FAISS system stored and searched vectors. But it only stored vectors — the actual document text was kept separately in a Python list.

What problems does this separation cause in a real application? Think about what happens if you restart the script, or if you add new documents later.